# 05. Decision Model Training (Ensemble)

This notebook builds the final decision models (Logicstic Regression and XGBoost) that ingest all raw data plus the engineered Alternative Credit Score.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import pickle
import os

df = pd.read_csv('../backend/data/synthetic_loan_data.csv')

# 1. Feature Engineering (Proxy for ACS)
def compute_acs(row):
    # Mimic the weighted logic from Model A and Model B
    pri = row['prior_repayment_record']
    util = row['utility_payment_consistency'] if pd.notnull(row['utility_payment_consistency']) else 0.6
    elec = row['electricity_payment_regularity'] if pd.notnull(row['electricity_payment_regularity']) else 0.6
    
    if pd.isnull(pri):
        score_a = min((util * 0.55 + elec * 0.45) * 100, 70)
    else:
        score_a = (util * 0.40 + elec * 0.30 + pri * 0.30) * 100
        
    # Simple income proxy for Model B component
    score_b = (np.log1p(row['applicant_income']) / 15) * 100 # Normalized scale
    
    return (score_a * 0.6) + (score_b * 0.4)

df['alternative_credit_score'] = df.apply(compute_acs, axis=1)

# 2. Preprocessing
cat_cols = ['gender', 'marital_status', 'education', 'employment_type', 'area_type', 'loan_purpose', 'credit_category', 'govt_socioeconomic_category']
num_cols = ['age', 'applicant_income', 'coapplicant_income', 'loan_amount', 'loan_term_months', 'dependents', 'electricity_bill_avg', 'electricity_payment_regularity', 'mobile_recharge_avg', 'mobile_recharge_frequency', 'utility_payment_consistency', 'prior_repayment_record', 'alternative_credit_score']

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

df[num_cols] = df[num_cols].fillna(df[num_cols].mean())

X = df[cat_cols + num_cols]
y = df['loan_repaid']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)

# 3. Training
lr = LogisticRegression()
lr.fit(X_res, y_res)

xgb = XGBClassifier()
xgb.fit(X_res, y_res)

# 4. Saving Artifacts
os.makedirs('../models', exist_ok=True)
with open('../models/lr_decision.pkl', 'wb') as f: pickle.dump(lr, f)
with open('../models/xgb_decision.pkl', 'wb') as f: pickle.dump(xgb, f)
with open('../models/scaler.pkl', 'wb') as f: pickle.dump(scaler, f)
with open('../models/label_encoders.pkl', 'wb') as f: pickle.dump(encoders, f)

print("Artifacts saved: lr_decision.pkl, xgb_decision.pkl, scaler.pkl, label_encoders.pkl")